# Schema Interconnectivity and Mapping Coverage in the Life-Sciences Linked Open Data Cloud

> **Production notebook** - loads pre-built Parquet artefacts from the pipeline.
> Do **not** edit data-loading cells manually; re-run the pipeline to update data.
> See `notebooks/mapping_analysis.ipynb` for the exploratory/dev version.

**Scope:**
- Schema-level connectivity (typed-object cross-dataset edges)
- Curated mapping layers (SSSOM, SeMRA, instance-based)
- Predicate relaxation
- Inference expansion
- Community structure (Louvain), centrality, reachability

**Four graph variants:**
- **G_schema** - typed-object edges from mined schemas only
- **G_raw** - G_schema + class-level + instance-resolved mappings
- **G_relaxed** - G_raw + predicate-relaxed edges
- **G_inferred** - G_relaxed + transitive/inverse closure

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import random
from pathlib import Path

import matplotlib.cm as cm
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from networkx.algorithms.community import louvain_communities

# ── paths ────────────────────────────────────────────────────────────────────
# notebook lives in notebooks/paper/ -> two levels up is repo root
ROOT       = Path("../..").resolve()
PAPER_DATA = ROOT / "results" / "paper_data"
GRAPHS_DIR = PAPER_DATA / "graphs"
FIGURES    = ROOT / "docs" / "paper" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

# ── matplotlib style ─────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})
sns.set_palette("muted")

# ── helpers ───────────────────────────────────────────────────────────────────
def _shorten(uri: str) -> str:
    """Return a compact label for a URI (last path/fragment segment)."""
    for sep in ("#", "/"):
        idx = uri.rfind(sep)
        if idx >= 0 and idx < len(uri) - 1:
            return uri[idx + 1:]
    return uri

def savefig(name: str) -> None:
    """Save current figure to docs/paper/figures/ as PDF and PNG."""
    for ext in ("pdf", "png"):
        plt.savefig(FIGURES / f"{name}.{ext}", bbox_inches="tight", dpi=150)
    print(f"  -> saved {name}.pdf / .png")

print(f"PAPER_DATA : {PAPER_DATA}")
print(f"FIGURES    : {FIGURES}")
print(f"PAPER_DATA exists: {PAPER_DATA.exists()}")

## 2. Pipeline metadata & benchmarks

In [ ]:
meta_path = PAPER_DATA / "metadata.json"
bench_path = PAPER_DATA / "benchmarks.jsonl"

if meta_path.exists():
    meta = json.loads(meta_path.read_text())
    print("Pipeline metadata:")
    for k, v in meta.items():
        print(f"  {k:<30s}: {v}")
else:
    print(f"WARNING: {meta_path} not found - run the pipeline first.")
    meta = {}

if bench_path.exists():
    bench_rows = [json.loads(line) for line in bench_path.read_text().splitlines() if line.strip()]
    df_bench = pd.DataFrame(bench_rows)
    print(f"\nBenchmark records: {len(df_bench)}")
    print(df_bench.to_string(index=False))
else:
    print(f"WARNING: {bench_path} not found.")
    df_bench = pd.DataFrame()

## 3. Load graphs from Parquet

All graphs are pre-built by `scripts/build_graphs.py` and stored as Parquet edge lists.
Loading is instantaneous- no JSON-LD scanning at notebook time.

In [ ]:
def load_graph(name: str) -> nx.Graph:
    """Load a dataset-level nx.Graph from a Parquet edge list."""
    path = GRAPHS_DIR / f"edges_{name}.parquet"
    if not path.exists():
        print(f"  WARNING: {path.name} not found - returning empty graph.")
        return nx.Graph()
    df = pd.read_parquet(path)
    G = nx.Graph()
    for row in df.itertuples(index=False):
        G.add_edge(row.dataset_a, row.dataset_b, weight=row.weight)
    return G

G_schema   = load_graph("schema")
G_raw      = load_graph("raw")
G_relaxed  = load_graph("relaxed")   # may be empty until L2e is complete
G_inferred = load_graph("inferred")

# Node metadata
nodes_path = GRAPHS_DIR / "nodes.parquet"
df_nodes = pd.read_parquet(nodes_path) if nodes_path.exists() else pd.DataFrame()

for label, G in [("G_schema", G_schema), ("G_raw", G_raw),
                 ("G_relaxed", G_relaxed), ("G_inferred", G_inferred)]:
    print(f"{label:<12s}: {G.number_of_nodes():3d} nodes, {G.number_of_edges():5d} edges")

## 4. Artefact inventory

In [ ]:
# Inventory from pipeline metadata / benchmarks
if not df_bench.empty and "step" in df_bench.columns:
    inv_steps = ["sssom_import", "semra_import", "instance_resolve",
                 "predicate_relax", "inference"]
    df_inv = df_bench[df_bench["step"].isin(inv_steps)].copy()
    print(df_inv[["step","files","edges","applied","elapsed_s"]]
          .to_string(index=False))
else:
    print("No benchmark data - run the pipeline first.")

## 5. Mapping inventory: edges by strategy and predicate

Pre-computed edge counts by strategy and predicate type are read from pipeline benchmarks.

In [ ]:
# Read strategy / predicate counts from benchmarks or nodes parquet
strat_path = PAPER_DATA / "strategy_counts.parquet"
pred_path  = PAPER_DATA / "predicate_counts.parquet"

if strat_path.exists() and pred_path.exists():
    strat_series = pd.read_parquet(strat_path).set_index("strategy")["count"].sort_values(ascending=False)
    pred_df      = pd.read_parquet(pred_path).sort_values("count", ascending=False)
    pred_series  = pred_df.set_index("predicate_short")["count"]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    strat_series.plot(kind="bar", ax=axes[0], color="steelblue", edgecolor="white")
    axes[0].set_title("Mapping edges by strategy")
    axes[0].set_ylabel("Edge count")
    axes[0].tick_params(axis="x", rotation=30)

    pred_series.head(15).plot(kind="barh", ax=axes[1], color="coral", edgecolor="white")
    axes[1].set_title("Top 15 mapping predicates")
    axes[1].set_xlabel("Edge count")
    axes[1].invert_yaxis()

    plt.tight_layout()
    savefig("fig_strategy_distribution")
    plt.show()
else:
    print("strategy/predicate counts not yet built - run build_graphs.py")

## 6. Connectivity metrics

In [ ]:
def graph_metrics(G: nx.Graph, label: str) -> dict:
    if G.number_of_nodes() == 0:
        return {"graph": label, "nodes": 0, "edges": 0,
                "components": 0, "lcc_nodes": 0, "diameter": 0, "density": 0.0}
    components = list(nx.connected_components(G))
    lcc = max(components, key=len)
    lcc_sub = G.subgraph(lcc)
    return {
        "graph":      label,
        "nodes":      G.number_of_nodes(),
        "edges":      G.number_of_edges(),
        "components": len(components),
        "lcc_nodes":  len(lcc),
        "diameter":   nx.diameter(lcc_sub) if len(lcc) > 1 else 0,
        "density":    round(nx.density(G), 4),
    }

df_metrics = pd.DataFrame([
    graph_metrics(G_schema,   "G_schema"),
    graph_metrics(G_raw,      "G_raw"),
    graph_metrics(G_relaxed,  "G_relaxed"),
    graph_metrics(G_inferred, "G_inferred"),
]).set_index("graph")
print(df_metrics.to_string())
df_metrics

## 7. Hidden bridges

Edges in G_inferred absent from G_raw - dataset pairs joinable only via inference.

In [ ]:
raw_edges      = {frozenset(e) for e in G_raw.edges()}
inferred_edges = {frozenset(e) for e in G_inferred.edges()}
new_bridges    = inferred_edges - raw_edges

print(rf"Hidden bridges (G_inferred \ G_raw): {len(new_bridges)}")
if new_bridges:
    rows = []
    for fs in sorted(new_bridges):
        u, v = tuple(fs)
        rows.append({"dataset_A": u, "dataset_B": v,
                     "inferred_pairs": G_inferred[u][v]["weight"]})
    df_bridges = (pd.DataFrame(rows)
                  .sort_values("inferred_pairs", ascending=False)
                  .reset_index(drop=True))
    print(df_bridges.to_string(index=False))

## 8. Hub analysis - betweenness centrality

In [ ]:
bc     = nx.betweenness_centrality(G_inferred, weight="weight", normalized=True)
degree = dict(G_inferred.degree(weight="weight"))

df_centrality = (
    pd.DataFrame({"dataset": list(bc.keys()), "betweenness": list(bc.values())})
    .assign(degree=lambda d: d["dataset"].map(degree))
    .sort_values("betweenness", ascending=False)
    .reset_index(drop=True)
)
print("Top 20 datasets by betweenness centrality (G_inferred):")
print(df_centrality.head(20).to_string(index=False))

# Hub classes from nodes parquet
if not df_nodes.empty and "class_uri" in df_nodes.columns:
    hub_classes = (df_nodes[df_nodes["n_datasets"] > 1]
                   .sort_values("n_datasets", ascending=False)
                   .head(20))
    print("\nTop hub classes (spanning most datasets):")
    print(hub_classes[["class_uri","n_datasets"]].to_string(index=False))

## 9. Community detection (Louvain) - G_schema

In [ ]:
random.seed(42)

G_conn_schema = G_schema.copy()
G_conn_schema.remove_nodes_from(list(nx.isolates(G_schema)))

comm_schema = louvain_communities(G_conn_schema, weight="weight", seed=42)
comm_schema_sorted = sorted(comm_schema, key=len, reverse=True)
n2c_schema = {n: i for i, c in enumerate(comm_schema_sorted) for n in c}

print(f"G_schema  - Louvain communities: {len(comm_schema_sorted)}")
print(f"Modularity: {nx.community.modularity(G_conn_schema, comm_schema_sorted, weight='weight'):.4f}")
for i, c in enumerate(comm_schema_sorted):
    members = sorted(c)
    print(f"  [{i:2d}] ({len(c):3d}): {', '.join(members[:8])}" + (" ..." if len(members) > 8 else ""))

### Connectivity visualisation - G_schema

In [ ]:
n_comm  = len(comm_schema_sorted)
cmap    = cm.get_cmap("tab20", max(n_comm, 1))
pos     = nx.spring_layout(G_conn_schema, weight="weight", seed=42, k=2.5)

fig, ax = plt.subplots(figsize=(20, 20))
nx.draw_networkx_edges(
    G_conn_schema, pos,
    width=[0.3 + 0.5 * np.log1p(G_conn_schema[u][v].get("weight", 1))
           for u, v in G_conn_schema.edges()],
    alpha=0.35, edge_color="grey", ax=ax)
nx.draw_networkx_nodes(
    G_conn_schema, pos,
    node_size=[50 + 10 * G_conn_schema.degree(n, weight="weight") for n in G_conn_schema.nodes()],
    node_color=[cmap(n2c_schema.get(n, 0)) for n in G_conn_schema.nodes()],
    alpha=0.85, ax=ax)
nx.draw_networkx_labels(G_conn_schema, pos, font_size=6, font_color="black", ax=ax)
ax.set_title(
    f"G_schema - {G_conn_schema.number_of_nodes()} datasets, "
    f"{G_conn_schema.number_of_edges()} edges, {n_comm} communities", fontsize=12)
ax.axis("off")
plt.tight_layout()
savefig("fig_community_gschema")
plt.show()

## 10. Community detection (Louvain) - G_inferred

In [ ]:
random.seed(42)

G_conn_inf = G_inferred.copy()
G_conn_inf.remove_nodes_from(list(nx.isolates(G_inferred)))

comm_inf = louvain_communities(G_conn_inf, weight="weight", seed=42)
communities_sorted = sorted(comm_inf, key=len, reverse=True)
node_to_community  = {n: i for i, c in enumerate(communities_sorted) for n in c}

print(f"G_inferred - Louvain communities: {len(communities_sorted)}")
print(f"Modularity: {nx.community.modularity(G_conn_inf, communities_sorted, weight='weight'):.4f}")
for i, c in enumerate(communities_sorted):
    members = sorted(c)
    print(f"  [{i:2d}] ({len(c):3d}): {', '.join(members[:8])}" + (" ..." if len(members) > 8 else ""))

isolated = sorted(nx.isolates(G_inferred))
print(f"\nIsolated datasets ({len(isolated)}): {', '.join(isolated)}")

### Connectivity visualisation - G_inferred

In [ ]:
n_comm = len(communities_sorted)
cmap   = cm.get_cmap("tab20", max(n_comm, 1))
pos    = nx.spring_layout(G_conn_inf, weight="weight", seed=42, k=2.5)

fig, ax = plt.subplots(figsize=(20, 20))
nx.draw_networkx_edges(
    G_conn_inf, pos,
    width=[0.3 + 0.5 * np.log1p(G_conn_inf[u][v].get("weight", 1))
           for u, v in G_conn_inf.edges()],
    alpha=0.35, edge_color="grey", ax=ax)
nx.draw_networkx_nodes(
    G_conn_inf, pos,
    node_size=[50 + 10 * G_conn_inf.degree(n, weight="weight") for n in G_conn_inf.nodes()],
    node_color=[cmap(node_to_community.get(n, 0)) for n in G_conn_inf.nodes()],
    alpha=0.85, ax=ax)
nx.draw_networkx_labels(G_conn_inf, pos, font_size=6, font_color="black", ax=ax)
ax.set_title(
    f"G_inferred - {G_conn_inf.number_of_nodes()} datasets, "
    f"{G_conn_inf.number_of_edges()} edges, {n_comm} communities", fontsize=12)
ax.axis("off")
plt.tight_layout()
savefig("fig_community_ginferred")
plt.show()

## 11. Dataset × dataset reachability heatmap

In [ ]:
top_ds = df_centrality.head(min(30, len(df_centrality)))["dataset"].tolist()

if len(top_ds) > 1:
    adj     = nx.to_pandas_adjacency(G_inferred.subgraph(top_ds), weight="weight").astype(float)
    adj_log = np.log1p(adj)

    fig, ax = plt.subplots(figsize=(14, 11))
    sns.heatmap(
        adj_log, ax=ax, cmap="YlOrRd", linewidths=0.3, linecolor="white",
        xticklabels=True, yticklabels=True,
        cbar_kws={"label": "log(1 + bridging class-pair count)"},
    )
    ax.set_title(
        "Log-scaled cross-dataset mapping weight\n"
        "(top datasets by betweenness centrality, G_inferred)", fontsize=11)
    plt.xticks(rotation=45, ha="right", fontsize=7)
    plt.yticks(rotation=0, fontsize=7)
    plt.tight_layout()
    savefig("fig_heatmap")
    plt.show()

## 12. Schema structural profile

In [ ]:
if not df_nodes.empty:
    df_ns = df_nodes.copy()

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    counts = df_ns["pattern_count"].clip(lower=1)
    log_bins = np.logspace(np.log10(counts.min()), np.log10(counts.max()), 30)
    axes[0].hist(counts, bins=log_bins, color="steelblue", edgecolor="white")
    axes[0].set_xscale("log")
    axes[0].set_title("Distribution of pattern counts per dataset")
    axes[0].set_xlabel("Pattern count (log scale)")
    axes[0].set_ylabel("Number of datasets")
    axes[0].xaxis.set_major_formatter(
        mticker.FuncFormatter(lambda val, pos: f"{int(val):,}"))
    axes[0].tick_params(axis="x", rotation=45)

    axes[1].hist(df_ns["class_count"], bins=30, color="seagreen", edgecolor="white")
    axes[1].set_title("Distribution of class counts per dataset")
    axes[1].set_xlabel("Number of distinct subject classes")
    axes[1].set_ylabel("Number of datasets")

    plt.tight_layout()
    savefig("fig_schema_profile")
    plt.show()

    print("Largest schemas by pattern count:")
    print(df_ns.sort_values("pattern_count", ascending=False).head(15).to_string(index=False))
else:
    print("Node metadata not yet built - run build_graphs.py")

## 13. Pipeline benchmark visualisation

In [ ]:
if not df_bench.empty and "step" in df_bench.columns and "elapsed_s" in df_bench.columns:
    df_time = (df_bench[["step","elapsed_s"]]
               .dropna(subset=["elapsed_s"])
               .sort_values("elapsed_s", ascending=True))

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(df_time["step"], df_time["elapsed_s"], color="steelblue", edgecolor="white")
    ax.set_xlabel("Wall-clock time (s)")
    ax.set_title("Pipeline step elapsed times")
    plt.tight_layout()
    savefig("fig_pipeline_benchmarks")
    plt.show()
else:
    print("No benchmark timing data available.")

## 14. Summary table

In [ ]:
schema_edge_set   = {frozenset(e) for e in G_schema.edges()}
raw_edge_set      = {frozenset(e) for e in G_raw.edges()}
relaxed_edge_set  = {frozenset(e) for e in G_relaxed.edges()}
inferred_edge_set = {frozenset(e) for e in G_inferred.edges()}

def _comp(G): return len(list(nx.connected_components(G))) if G.number_of_nodes() else "-"

summary = {
    "Pipeline run date":              meta.get("run_date", "-"),
    "rdfsolve version":               meta.get("rdfsolve_version", "-"),
    "Mined schemas":                  df_nodes["dataset"].nunique() if not df_nodes.empty else "-",
    "G_schema nodes / edges":         f"{G_schema.number_of_nodes()} / {G_schema.number_of_edges()}",
    "G_raw nodes / edges":            f"{G_raw.number_of_nodes()} / {G_raw.number_of_edges()}",
    "G_relaxed nodes / edges":        f"{G_relaxed.number_of_nodes()} / {G_relaxed.number_of_edges()}",
    "G_inferred nodes / edges":       f"{G_inferred.number_of_nodes()} / {G_inferred.number_of_edges()}",
    "G_schema components":            _comp(G_schema),
    "G_raw components":               _comp(G_raw),
    "G_inferred components":          _comp(G_inferred),
    r"New pairs raw \ schema":         len(raw_edge_set - schema_edge_set),
    r"New pairs relaxed \ raw":        len(relaxed_edge_set - raw_edge_set),
    r"New pairs inferred \ relaxed":   len(inferred_edge_set - relaxed_edge_set),
    "Louvain communities (G_inferred)": len(communities_sorted),
    "Isolated datasets":              len(isolated),
}

for k, v in summary.items():
    print(f"  {k:<45s}: {v}")